# Analisis de deriva latente pre/post fine-tuning

Compara cuantitativamente el espacio latente z0 antes y despues del fine-tuning:

- **Deriva geometrica**: separabilidad, PR, desplazamiento medio.
- **Rotacion PCA**: alineacion Grassmann entre subespacios principales.
- **Deriva semantica**: norma, consistencia y ortogonalidad de direcciones de atributo.

**Puede ejecutarse sin GPU** (opera sobre tensores precalculados).

**Prerequisitos:**
- `corpus/latents/{model}/`
- `ajuste_fino/latents_post_ft/{model}/`

---
## 0 — Entorno

In [ ]:
from google.colab import drive
import os, sys, json
from datetime import datetime

drive.mount("/content/drive", force_remount=False)
PROJECT_ROOT = "/content/drive/MyDrive/TFM/"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

LATENTS_DIR    = os.path.join(PROJECT_ROOT, "data", "latents", "latents")
LATENTS_FT_DIR = os.path.join(PROJECT_ROOT, "data", "latents", "latents_ft")
RESULTS_DIR    = os.path.join(PROJECT_ROOT, "data", "fase5", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

MODELS       = ["sd15", "sd21", "sdxl"]
MODEL_COLORS = {"sd15": "#4C72B0", "sd21": "#DD8452", "sdxl": "#55A868"}
ATTR_TYPES   = ["color", "lighting", "style"]
ATTR_PFX     = {"color": "color", "lighting": "light", "style": "style"}

print("Entorno listo.")
print(f"Resultados -> {RESULTS_DIR}")

---
## 1 — Carga de datos

In [ ]:
def load_latents(base_dir, model):
    path = os.path.join(base_dir, model, "latents.pt")
    mf   = os.path.join(base_dir, model, "manifest.json")
    if not os.path.exists(path):
        return None, None
    tensor = torch.load(path, map_location="cpu", weights_only=False).float()
    if tensor.ndim == 5:
        tensor = tensor.squeeze(1)
    N = tensor.shape[0]
    Z = tensor.numpy().reshape(N, -1)
    with open(mf) as f:
        manifest = json.load(f)
    return Z, manifest

available = []
for m in MODELS:
    Z_pre, _ = load_latents(LATENTS_DIR, m)
    Z_ft,  _ = load_latents(LATENTS_FT_DIR, m)
    if Z_pre is not None and Z_ft is not None:
        available.append(m)
        print(f"  {m.upper()} : pre {Z_pre.shape}  ft {Z_ft.shape}")
    else:
        missing = []
        if Z_pre is None: missing.append("pre (Fase 2)")
        if Z_ft  is None: missing.append("ft (fase5_latents)")
        print(f"  {m.upper()} : SKIP — faltan {', '.join(missing)}")

print(f"\nModelos disponibles: {available}")

---
## 2 — Funciones de analisis

In [ ]:
# Funciones de análisis de deriva latente. Cada función implementa una
# métrica del protocolo de comparación pre/post fine-tuning:
#
# dist_matrix: matriz de distancias L2 con la identidad ‖x-y‖²=‖x‖²+‖y‖²-2xᵀy.
# intra_inter: separa distancias en intra-prompt (misma semántica) e inter-prompt.
# participation_ratio: PR=(Σλ)²/Σλ², número efectivo de dimensiones activas.
# pca_basis: extrae los k primeros vectores singulares derechos (base PCA).
# grassmann_align: alineación entre subespacios PCA via ‖AᵀB‖_F²/k ∈ [0,1].
#   Valor 1 = subespacios idénticos; valores bajos indican rotación de la base.
# semantic_dirs: reconstruye las direcciones semánticas pretrain/post como
#   diferencia de centroides, usando los mismos grupos que en la Fase 4.
# intra_consistency: coseno medio intra-tipo como proxy de universalidad.
def dist_matrix(Z):
    sq = (Z ** 2).sum(1)
    d2 = np.clip(sq[:, None] + sq[None, :] - 2 * (Z @ Z.T), 0, None)
    return np.sqrt(d2)

def intra_inter(D, manifest):
    N = len(manifest)
    intra, inter = [], []
    for i in range(N):
        for j in range(i + 1, N):
            same = manifest[i]["id"] == manifest[j]["id"]
            (intra if same else inter).append(D[i, j])
    return np.array(intra), np.array(inter)

def participation_ratio(Z):
    Z_c     = Z - Z.mean(0)
    _, S, _ = np.linalg.svd(Z_c, full_matrices=False)
    lam     = (S ** 2) / max(len(Z) - 1, 1)
    lam     = lam[lam > 0]
    return float(lam.sum() ** 2 / (lam ** 2).sum())

def pca_basis(Z, k=50):
    Z_c      = Z - Z.mean(0)
    _, _, Vt = np.linalg.svd(Z_c, full_matrices=False)
    return Vt[:k]

def grassmann_align(Va, Vb, k=10):
    k    = min(k, Va.shape[0], Vb.shape[0])
    dots = (Va[:k] @ Vb[:k].T) ** 2
    return float(dots.sum(axis=1).mean())

def pca_energy(direction, V_basis, k=10):
    # Fracción de energía en los top-k PCs, normalizada por la energía total
    # capturada en todos los K componentes de V_basis (K=50 por defecto).
    # Rango [0, 1]: 1 = la dirección vive completamente en los top-k componentes.
    d_norm  = direction / (np.linalg.norm(direction) + 1e-8)
    projs   = V_basis @ d_norm             # proyecciones en los K componentes
    e_top   = float((projs[:k] ** 2).sum())
    e_total = float((projs ** 2).sum())
    return e_top / (e_total + 1e-8)

def semantic_dirs(Z, manifest):
    groups = {}
    for e in manifest:
        if e.get("category") != "attribute":
            continue
        g, v = e["group"], e["attribute_value"]
        groups.setdefault(g, {}).setdefault(v, []).append(Z[e["idx"]])
    dirs = {}
    for g, vd in groups.items():
        if len(vd) < 2:
            continue
        sv = sorted(vd.keys())
        dirs[g] = np.mean(vd[sv[1]], axis=0) - np.mean(vd[sv[0]], axis=0)
    return dirs

def intra_consistency(dirs, pfx):
    ds = [d for g, d in dirs.items() if g.startswith(pfx)]
    if len(ds) < 2:
        return 0.0
    ns  = [d / (np.linalg.norm(d) + 1e-8) for d in ds]
    cos = [float(np.dot(ns[i], ns[j]))
           for i in range(len(ns)) for j in range(i + 1, len(ns))]
    return float(np.mean(cos))

def inter_entanglement(dirs, p1, p2):
    d1 = [d / (np.linalg.norm(d) + 1e-8) for g, d in dirs.items() if g.startswith(p1)]
    d2 = [d / (np.linalg.norm(d) + 1e-8) for g, d in dirs.items() if g.startswith(p2)]
    if not d1 or not d2:
        return 0.0
    return float(np.mean([abs(np.dot(a, b)) for a in d1 for b in d2]))

print("Funciones de analisis definidas.")

---
## 3 — Analisis por modelo

In [ ]:
# Análisis de deriva por modelo. Para cada modelo calcula en paralelo:
#   1. Deriva geométrica: sep, PR, coseno entre centroides, desplazamiento
#      medio ‖z_ft-z_pre‖ y test KS sobre las distribuciones de distancias.
#   2. Rotación PCA: alineación Grassmann entre los subespacios top-10/50
#      pretrain y post-fine-tuning. Cuantifica cuánto ha rotado la base.
#   3. Deriva semántica: consistencia intra-tipo y energía en PCA para los
#      tres atributos (color, lighting, style), pre y post fine-tuning.
# Los resultados se almacenan en all_results con claves privadas (_ip, _V_pre…)
# que se excluyen al serializar a JSON pero se usan para generar las figuras.
all_results = {}

for model in available:
    print(f"\n  [{model.upper()}]")
    Z_pre, mf = load_latents(LATENTS_DIR, model)
    Z_ft,  _  = load_latents(LATENTS_FT_DIR, model)

    mask_base = np.array([e["category"] == "base" for e in mf])
    Z_pre_b   = Z_pre[mask_base]
    Z_ft_b    = Z_ft[mask_base]
    mf_b      = [e for e in mf if e["category"] == "base"]

    D_pre = dist_matrix(Z_pre_b); D_ft = dist_matrix(Z_ft_b)
    ip, ep = intra_inter(D_pre, mf_b)
    if_, ef = intra_inter(D_ft,  mf_b)
    sep_pre = float(ep.mean() / (ip.mean() + 1e-8))
    sep_ft  = float(ef.mean() / (if_.mean() + 1e-8))
    pr_pre  = participation_ratio(Z_pre_b)
    pr_ft   = participation_ratio(Z_ft_b)
    c_pre   = Z_pre_b.mean(0); c_ft = Z_ft_b.mean(0)
    cos_c   = float(np.dot(c_pre, c_ft) /
                    (np.linalg.norm(c_pre) * np.linalg.norm(c_ft) + 1e-8))
    deltas  = np.linalg.norm(Z_ft - Z_pre, axis=1)
    ks_stat, ks_p = ks_2samp(ip, if_)
    print(f"    sep  pre={sep_pre:.3f}  ft={sep_ft:.3f}  d={sep_ft-sep_pre:+.3f}")
    print(f"    PR   pre={pr_pre:.1f}   ft={pr_ft:.1f}   d={pr_ft-pr_pre:+.1f}")
    print(f"    delta_lat: mu={deltas.mean():.2f}  std={deltas.std():.2f}")

    V_pre   = pca_basis(Z_pre_b); V_ft = pca_basis(Z_ft_b)
    align10 = grassmann_align(V_pre, V_ft, 10)
    align50 = grassmann_align(V_pre, V_ft, 50)
    print(f"    PCA  align10={align10:.3f}  align50={align50:.3f}")

    dirs_pre = semantic_dirs(Z_pre, mf)
    dirs_ft  = semantic_dirs(Z_ft,  mf)
    sem = {}
    for attr in ATTR_TYPES:
        pfx = ATTR_PFX[attr]
        n_pre = float(np.mean([np.linalg.norm(d) for g,d in dirs_pre.items() if g.startswith(pfx)] or [0]))
        n_ft  = float(np.mean([np.linalg.norm(d) for g,d in dirs_ft.items()  if g.startswith(pfx)] or [0]))
        c_p   = intra_consistency(dirs_pre, pfx)
        c_f   = intra_consistency(dirs_ft,  pfx)
        e_p   = float(np.mean([pca_energy(dirs_pre[g], V_pre) for g in dirs_pre if g.startswith(pfx)] or [0]))
        e_f   = float(np.mean([pca_energy(dirs_ft[g],  V_ft)  for g in dirs_ft  if g.startswith(pfx)] or [0]))
        sem[attr] = {"norm_pre": n_pre, "norm_ft": n_ft,
                     "consistency_pre": c_p, "consistency_ft": c_f,
                     "energy_pre": e_p, "energy_ft": e_f}
        print(f"    {attr:<8}  cons {c_p:.3f}->{c_f:.3f} ({c_f-c_p:+.3f})")

    all_results[model] = {
        "geometry": {"sep_pre": sep_pre, "sep_ft": sep_ft, "delta_sep": sep_ft-sep_pre,
                     "pr_pre": pr_pre, "pr_ft": pr_ft, "delta_pr": pr_ft-pr_pre,
                     "centroid_cos": cos_c, "delta_mean": float(deltas.mean()),
                     "ks_stat": float(ks_stat), "ks_p": float(ks_p)},
        "pca": {"alignment_top10": align10, "alignment_top50": align50},
        "semantics": sem,
        "_ip": ip, "_ep": ep, "_if": if_, "_ef": ef, "_deltas": deltas,
        "_V_pre": V_pre, "_V_ft": V_ft, "_dirs_pre": dirs_pre, "_dirs_ft": dirs_ft,
    }

print("\nAnalisis completado:", available)

---
## 4 — Figuras por modelo

In [ ]:
# Figura de 6 paneles (2×3) por modelo: fila superior con métricas geométricas
# (distribución distancias, sep/PR normalizados, histograma de desplazamiento)
# y fila inferior con métricas semánticas (consistencia intra-tipo, energía PCA
# en top-10 PCs, curva de alineación Grassmann en función del top-k).
# Cada figura se guarda en ajuste_fino/resultados/deriva_{model}.png para el manuscrito.
def plot_model(r, model):
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f"{model.upper()} — Deriva latente pre/post fine-tuning (WikiArt)", fontsize=13)
    c = MODEL_COLORS[model]; g = r["geometry"]; sm = r["semantics"]

    ax = axes[0][0]
    ax.hist(r["_ip"], bins=35, alpha=0.7, color=c,      density=True, label="Intra pre")
    ax.hist(r["_if"], bins=35, alpha=0.0, color=c,      density=True, label="Intra post", histtype="step", lw=2)
    ax.hist(r["_ep"], bins=35, alpha=0.4, color="gray", density=True, label="Inter pre")
    ax.hist(r["_ef"], bins=35, alpha=0.0, color="gray", density=True, label="Inter post", histtype="step", lw=2)
    ax.set_xlabel("Distancia L2"); ax.set_title("Dist. intra/inter (pre/post)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[0][1]
    vals = [g["sep_pre"], g["sep_ft"], g["pr_pre"]/15, g["pr_ft"]/15]
    lbls = ["Sep pre", "Sep post", "PR/15 pre", "PR/15 post"]
    col4 = [c, c, "#E67E22", "#E67E22"]; alp4 = [0.9, 0.5, 0.9, 0.5]
    bars = ax.bar(range(4), vals, color=col4)
    for b, a in zip(bars, alp4): b.set_alpha(a)
    ax.set_xticks(range(4)); ax.set_xticklabels(lbls, fontsize=8)
    ax.axhline(1, color="red", lw=1, ls="--", alpha=0.5)
    for i, v in enumerate(vals): ax.text(i, v+0.01, f"{v:.2f}", ha="center", fontsize=9)
    ax.set_title("Separabilidad y PR/15"); ax.grid(axis="y", alpha=0.3)

    ax = axes[0][2]
    ax.hist(r["_deltas"], bins=35, color=c, alpha=0.8, density=True)
    ax.axvline(r["_deltas"].mean(), color="red", lw=2, ls="--", label=f"mu={r['_deltas'].mean():.1f}")
    ax.set_xlabel("||z_ft - z_pre||_2"); ax.set_title("Desplazamiento por latente")
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1][0]
    x = np.arange(len(ATTR_TYPES)); w = 0.35
    ax.bar(x-w/2, [sm[a]["consistency_pre"] for a in ATTR_TYPES], w, label="Pre",  color=c, alpha=0.9)
    ax.bar(x+w/2, [sm[a]["consistency_ft"]  for a in ATTR_TYPES], w, label="Post", color=c, alpha=0.5)
    ax.set_xticks(x); ax.set_xticklabels(ATTR_TYPES)
    ax.set_ylabel("Consistencia coseno intra-tipo"); ax.set_title("Consistencia semantica pre/post")
    ax.legend(); ax.grid(axis="y", alpha=0.3)

    ax = axes[1][1]
    ax.bar(x-w/2, [sm[a]["energy_pre"] for a in ATTR_TYPES], w, label="Pre",  color=c, alpha=0.9)
    ax.bar(x+w/2, [sm[a]["energy_ft"]  for a in ATTR_TYPES], w, label="Post", color=c, alpha=0.5)
    ax.set_xticks(x); ax.set_xticklabels(ATTR_TYPES)
    ax.set_ylabel("Fraccion de energia en top-10 PCs [0,1]"); ax.set_title("Energia PCA de direcciones semanticas")
    ax.legend(); ax.grid(axis="y", alpha=0.3)

    ax = axes[1][2]
    Va = r["_V_pre"]; Vb = r["_V_ft"]
    ks = list(range(1, 51))
    ax.plot(ks, [grassmann_align(Va, Vb, k) for k in ks], color=c, lw=2)
    ax.axhline(1.0, color="gray", lw=1, ls="--")
    ax.set_xlabel("Top-k componentes PCA"); ax.set_ylabel("Alineacion Grassmann")
    ax.set_title("Rotacion de la base PCA"); ax.set_xlim(1, 50); ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, f"deriva_{model}.png")
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Figura guardada: {out}")

for model in available:
    plot_model(all_results[model], model)

---
## 5 — Comparativa cross-model

In [ ]:
if len(available) >= 2:
    models = available; colors = [MODEL_COLORS[m] for m in models]
    x = np.arange(len(ATTR_TYPES)); w = 0.25

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle("Fase 5 — Comparativa cross-model de deriva latente (WikiArt)", fontsize=14)

    ax = axes[0][0]
    vals = [all_results[m]["geometry"]["delta_sep"] for m in models]
    bars = ax.bar(models, vals, color=colors, alpha=0.85)
    ax.axhline(0, color="black", lw=1)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v+0.002*(1 if v>=0 else -1), f"{v:+.3f}", ha="center", fontsize=10)
    ax.set_title("Delta Separabilidad"); ax.grid(axis="y", alpha=0.3)

    ax = axes[0][1]
    vals = [all_results[m]["geometry"]["delta_pr"] for m in models]
    bars = ax.bar(models, vals, color=colors, alpha=0.85)
    ax.axhline(0, color="black", lw=1)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v+0.05*(1 if v>=0 else -1), f"{v:+.1f}", ha="center", fontsize=10)
    ax.set_title("Delta Participation Ratio"); ax.grid(axis="y", alpha=0.3)

    ax = axes[0][2]
    vals = [all_results[m]["pca"]["alignment_top10"] for m in models]
    bars = ax.bar(models, vals, color=colors, alpha=0.85)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.3f}", ha="center", fontsize=10)
    ax.set_ylim(0, 1.1); ax.set_title("Alineacion PCA top-10 (1=sin rotacion)")
    ax.grid(axis="y", alpha=0.3)

    ax = axes[1][0]
    for i, (m, c) in enumerate(zip(models, colors)):
        dc = [all_results[m]["semantics"][a]["consistency_ft"] -
              all_results[m]["semantics"][a]["consistency_pre"] for a in ATTR_TYPES]
        ax.bar(x+(i-1)*w, dc, w, label=m.upper(), color=c, alpha=0.85)
    ax.axhline(0, color="black", lw=1)
    ax.set_xticks(x); ax.set_xticklabels(ATTR_TYPES)
    ax.set_title("Delta consistencia intra-tipo (+=mejora)"); ax.legend(); ax.grid(axis="y", alpha=0.3)

    ax = axes[1][1]
    for i, (m, c) in enumerate(zip(models, colors)):
        de = [all_results[m]["semantics"][a]["energy_ft"] -
              all_results[m]["semantics"][a]["energy_pre"] for a in ATTR_TYPES]
        ax.bar(x+(i-1)*w, de, w, label=m.upper(), color=c, alpha=0.85)
    ax.axhline(0, color="black", lw=1)
    ax.set_xticks(x); ax.set_xticklabels(ATTR_TYPES)
    ax.set_title("Delta fraccion energia PCA (+=mas concentrado)"); ax.legend(); ax.grid(axis="y", alpha=0.3)

    ax = axes[1][2]
    vals = [all_results[m]["geometry"]["delta_mean"] for m in models]
    bars = ax.bar(models, vals, color=colors, alpha=0.85)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v+0.5, f"{v:.1f}", ha="center", fontsize=10)
    ax.set_ylabel("||z_ft - z_pre||_2 medio"); ax.set_title("Desplazamiento latente medio")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, "deriva_cross_model.png")
    plt.savefig(out, dpi=120, bbox_inches="tight"); plt.show()
    print(f"Figura cross-model guardada: {out}")
else:
    print("Solo un modelo disponible; figura cross-model omitida.")

---
## 6 — Exportacion de resultados

In [ ]:
# Exportación del resumen numérico completo a JSON.
# make_serializable convierte arrays NumPy a tipos Python nativos y excluye
# las claves privadas (prefijo '_') que contienen arrays intermedios usados
# solo para las figuras. El JSON resultante es el input del análisis del
# manuscrito (§4.4 del TFM) y el archivo fase5_deriva_summary.json.
def make_serializable(d):
    if isinstance(d, dict):
        return {k: make_serializable(v) for k, v in d.items() if not k.startswith("_")}
    if isinstance(d, np.ndarray):
        return float(d) if d.ndim == 0 else d.tolist()
    if isinstance(d, (np.floating, np.integer)):
        return float(d)
    if isinstance(d, (list, tuple)):
        return [make_serializable(v) for v in d]
    return d

summary = {"fase": 5, "fecha": datetime.now().isoformat(),
           "modelos": {m: make_serializable(r) for m, r in all_results.items()}}

out = os.path.join(RESULTS_DIR, "fase5_deriva_summary.json")
with open(out, "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Resumen guardado: {out}")
print(f"Figuras en      : {RESULTS_DIR}")
print()
for m in available:
    g = all_results[m]["geometry"]; pa = all_results[m]["pca"]; sm = all_results[m]["semantics"]
    print(f"{m.upper()}:")
    print(f"  sep    : {g['sep_pre']:.3f} -> {g['sep_ft']:.3f}  ({g['delta_sep']:+.3f})")
    print(f"  PR     : {g['pr_pre']:.1f}  -> {g['pr_ft']:.1f}   ({g['delta_pr']:+.1f})")
    print(f"  PCA-10 : {pa['alignment_top10']:.3f}")
    for a in ATTR_TYPES:
        s = sm[a]
        print(f"  {a:<8}: cons {s['consistency_pre']:.3f} -> {s['consistency_ft']:.3f}")
    print()
print("=== Fase 5 completada ===")